# Data Visualisation: What Goes into the Model

Presentation figures that walk through the data from the radar scene to the patches the U-Net is trained on:

1. The study area and the flood label
2. Before and after the cyclone, and how the label is made
3. Why a 3.5 dB drop marks flooding
4. How the scene is split into train, validation and test
5. The patches the model actually sees
6. Normalisation
7. Augmentation

Every figure is saved as a PNG in `notebooks/figures/`.

**Before running:**
- Copy `CycloneGabrielle_HawkesBay_S1_Stack_v2.tif` from Google Drive into the repository's `data/` folder.
- `data/dataset_patches/` must exist (made by `data_preprocessing.ipynb`).
- Paths are relative to this notebook, so run it from `notebooks/`.

In [ ]:
# Cell 1: Install dependencies
%pip install -q rasterio matplotlib pandas

In [ ]:
import os
import glob
import json

import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.windows import Window
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
from matplotlib.patches import Patch, Rectangle

# -------------------------------------------------------------------------
# 1. Configuration
# -------------------------------------------------------------------------
# Paths are relative to this notebook (run it from notebooks/)
TIF_PATH = os.path.join('..', 'data', 'CycloneGabrielle_HawkesBay_S1_Stack_v2.tif')
DATA_DIR = os.path.join('..', 'data', 'dataset_patches')
FIG_DIR = 'figures'
os.makedirs(FIG_DIR, exist_ok=True)

STEP = 4                  # Scene views are read at 1/4 resolution (40 m pixels); zooms use full resolution
ZOOM_SIZE = 1024          # Zoom window in pixels (10.24 km)
PRE_DATE, POST_DATE = '21 Jan 2023', '14 Feb 2023'  # Acquisitions chosen in data_collection.ipynb
FLOOD_THRESHOLD_DB = -3.5 # The labelling rule from data_collection.ipynb

# -------------------------------------------------------------------------
# 2. Figure Style (one look across every slide)
# -------------------------------------------------------------------------
SURFACE, INK, INK_2, MUTED, GRID, AXIS = '#fcfcfb', '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
SERIES = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']  # Categorical colours, always used in this order
FLOOD = SERIES[0]
SPLIT_COLOURS = {'train': SERIES[0], 'val': SERIES[1], 'test': SERIES[2]}
SPLIT_NAMES = {'train': 'Train', 'val': 'Validation', 'test': 'Test'}

# Change in backscatter: blue = drop (the flood signal), grey = no change, red = rise
VV_CHANGE_CMAP = LinearSegmentedColormap.from_list(
    'vv_change', ['#104281', '#3987e5', '#f0efec', '#e66767', '#a42a2a'])
FLOOD_CMAP = ListedColormap([FLOOD])  # NaN pixels stay transparent

plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'font.family': 'sans-serif', 'font.sans-serif': ['Helvetica Neue', 'Arial', 'DejaVu Sans'],
    'font.size': 11, 'axes.titlesize': 13, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'text.color': INK, 'axes.labelcolor': INK_2, 'xtick.color': MUTED, 'ytick.color': MUTED,
    'axes.edgecolor': AXIS, 'axes.spines.top': False, 'axes.spines.right': False,
    'grid.color': GRID, 'grid.linewidth': 0.6, 'legend.frameon': False,
    'lines.linewidth': 2, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
})

def save(fig, name):
    fig.savefig(os.path.join(FIG_DIR, name))
    plt.show()

def image_axes(ax, title):
    ax.set_title(title, fontsize=12)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

In [ ]:
# -------------------------------------------------------------------------
# 3. Load the Scene (reduced resolution) & Dataset Settings
# -------------------------------------------------------------------------
with rasterio.open(TIF_PATH) as src:
    H, W, transform = src.height, src.width, src.transform
    names = list(src.descriptions)
    # Nearest-neighbour keeps the original pixel values, so the label rule still holds exactly
    scene = src.read(out_shape=(src.count, H // STEP, W // STEP), resampling=Resampling.nearest)

bands = {name: scene[i] for i, name in enumerate(names)}
valid = np.isfinite(scene).all(axis=0) & ~(scene[:4] == 0).all(axis=0)
flooded = valid & (bands['baseline_mask'] > 0.5)

# Map extent in km (UTM zone 60S), for axes that read as distances
extent_km = [transform.c / 1000, (transform.c + W * transform.a) / 1000,
             (transform.f + H * transform.e) / 1000, transform.f / 1000]

with open(os.path.join(DATA_DIR, 'dataset_info.json')) as f:
    dataset_info = json.load(f)

summary = {
    'Scene size (pixels)': f'{H} x {W}',
    'Pixel size': f'{transform.a:.0f} m',
    'Area': f'{H * W * transform.a ** 2 / 1e6:,.0f} km²',
    'Valid pixels': f'{100 * valid.mean():.1f}%',
    'Flood label': f'{100 * flooded.sum() / valid.sum():.2f}% of valid pixels',
    'Bands': ', '.join(names),
}
for key, value in summary.items():
    print(f'{key:<22}{value}')

## 1. The Study Area

Hawke's Bay and the Esk Valley after Cyclone Gabrielle. Radar backscatter is shown in grey; dark areas reflect little signal back to the satellite, which is typical of calm water. Blue marks the pixels labelled as flooded.

In [ ]:
# -------------------------------------------------------------------------
# 4. Figure: Study Area & Flood Label
# -------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 10))
ax.imshow(np.where(valid, bands['post_vv'], np.nan), cmap='gray', vmin=-25, vmax=0, extent=extent_km)
ax.imshow(np.where(flooded, 1.0, np.nan), cmap=FLOOD_CMAP, extent=extent_km, interpolation='nearest')
ax.set_title(f"Hawke's Bay after Cyclone Gabrielle, {POST_DATE}")
ax.set_xlabel('Easting (km, UTM zone 60S)')
ax.set_ylabel('Northing (km)')
ax.legend(handles=[Patch(color=FLOOD, label=f'Flood label: {100 * flooded.sum() / valid.sum():.1f}% of pixels')],
          loc='lower left', frameon=True, facecolor=SURFACE, edgecolor=AXIS)
save(fig, '01_study_area.png')

## 2. Before and After, and How the Label Is Made

A zoom on the most flooded part of the scene. Flooded ground turns dark after the cyclone, so the VV backscatter drops. The label is simply every pixel where it dropped by more than 3.5 dB, outside permanent water.

In [ ]:
# -------------------------------------------------------------------------
# 5. Figure: Before / After / Change / Label (full-resolution zoom)
# -------------------------------------------------------------------------
# Centre the zoom on the 512-pixel block with the most labelled flooding
block = 512 // STEP
counts = {(by, bx): flooded[by:by + block, bx:bx + block].sum()
          for by in range(0, flooded.shape[0], block) for bx in range(0, flooded.shape[1], block)}
by, bx = max(counts, key=counts.get)
size = min(ZOOM_SIZE, H, W)
y0 = int(np.clip((by + block // 2) * STEP - size // 2, 0, H - size))
x0 = int(np.clip((bx + block // 2) * STEP - size // 2, 0, W - size))

with rasterio.open(TIF_PATH) as src:
    zoom = {name: src.read(i + 1, window=Window(x0, y0, size, size)) for i, name in enumerate(names)}
zoom_flood = np.where(zoom['baseline_mask'] > 0.5, 1.0, np.nan)

fig, axes = plt.subplots(1, 4, figsize=(17, 4.8), constrained_layout=True)
db = axes[0].imshow(zoom['pre_vv'], cmap='gray', vmin=-25, vmax=0)
axes[1].imshow(zoom['post_vv'], cmap='gray', vmin=-25, vmax=0)
change = axes[2].imshow(zoom['vv_diff'], cmap=VV_CHANGE_CMAP, vmin=-10, vmax=10)
axes[3].imshow(zoom['post_vv'], cmap='gray', vmin=-25, vmax=0)
axes[3].imshow(zoom_flood, cmap=FLOOD_CMAP, interpolation='nearest')

image_axes(axes[0], f'Before ({PRE_DATE})')
image_axes(axes[1], f'After ({POST_DATE})')
image_axes(axes[2], 'Change in VV (after − before)')
image_axes(axes[3], f'Label: drop > {abs(FLOOD_THRESHOLD_DB)} dB')
fig.colorbar(db, ax=axes[:2], shrink=0.8, label='VV backscatter (dB)')
fig.colorbar(change, ax=axes[2], shrink=0.8, label='Change (dB)')
fig.suptitle(f'{size * transform.a / 1000:.1f} km × {size * transform.a / 1000:.1f} km zoom, full 10 m resolution',
             x=0.01, ha='left', color=INK_2, fontsize=11)
save(fig, '02_before_after_label.png')

## 3. Why a 3.5 dB Drop Marks Flooding

Left: across the whole scene, backscatter after the cyclone has a heavier dark tail than before. Right: almost every pixel changes by less than a few dB; the label takes the small tail that dropped further than the threshold (then removes permanent water). The y-axis is logarithmic because flooded pixels are so rare.

In [ ]:
# -------------------------------------------------------------------------
# 6. Figure: Backscatter & Change Distributions
# -------------------------------------------------------------------------
fig, (ax_db, ax_change) = plt.subplots(1, 2, figsize=(14, 4.8), constrained_layout=True)

db_bins = np.linspace(-30, 5, 141)
for name, label, colour in (('pre_vv', f'Before ({PRE_DATE})', SERIES[0]), ('post_vv', f'After ({POST_DATE})', SERIES[1])):
    ax_db.hist(bands[name][valid], bins=db_bins, histtype='step', linewidth=2, color=colour, label=label)
ax_db.set_title('VV backscatter, whole scene')
ax_db.set_xlabel('VV backscatter (dB)')
ax_db.set_ylabel('Pixels (40 m sample)')
ax_db.grid(axis='y')
ax_db.legend(loc='upper left')

diff = bands['vv_diff'][valid]
change_bins = np.linspace(-15, 15, 121)
counts, edges, _ = ax_change.hist(diff, bins=change_bins, color=AXIS, label='All pixels')
ax_change.hist(diff[diff < FLOOD_THRESHOLD_DB], bins=change_bins, color=FLOOD,
               label=f'Drop > {abs(FLOOD_THRESHOLD_DB)} dB ({100 * (diff < FLOOD_THRESHOLD_DB).mean():.1f}%)')
ax_change.axvline(FLOOD_THRESHOLD_DB, color=INK, linewidth=1.25, linestyle='--')
ax_change.annotate(f'{FLOOD_THRESHOLD_DB} dB threshold', xy=(FLOOD_THRESHOLD_DB, counts.max()),
                   xytext=(-8, 0), textcoords='offset points', ha='right', va='top', color=INK_2)
ax_change.set_yscale('log')
ax_change.set_title('Change in VV (after − before)')
ax_change.set_xlabel('Change (dB)')
ax_change.set_ylabel('Pixels (log scale)')
ax_change.legend(loc='upper right')
save(fig, '03_distributions.png')

## 4. Splitting the Scene

Neighbouring patches overlap by 50%, so splitting patch by patch would put near-copies in both training and test. Instead the scene is cut into 512-pixel blocks and each whole block goes to one split. Blocks were assigned so each split gets a similar share of the flooding.

In [ ]:
# -------------------------------------------------------------------------
# 7. Split Statistics (from the saved patches)
# -------------------------------------------------------------------------
split_stats = {}
for split in SPLIT_NAMES:
    masks = [np.load(p) for p in sorted(glob.glob(os.path.join(DATA_DIR, split, 'masks', '*.npy')))]
    split_stats[split] = {
        'Blocks': len(dataset_info['blocks'][split]),
        'Patches': len(masks),
        'Flooded pixels (%)': round(100 * np.mean([m.mean() for m in masks]), 2) if masks else 0.0,
    }
split_table = pd.DataFrame.from_dict(split_stats, orient='index').rename(index=SPLIT_NAMES)
split_table

In [ ]:
# -------------------------------------------------------------------------
# 8. Figure: Spatial Split Map & Split Sizes
# -------------------------------------------------------------------------
fig = plt.figure(figsize=(14, 10), constrained_layout=True)
grid = fig.add_gridspec(2, 2, width_ratios=[1.1, 1])
ax_map = fig.add_subplot(grid[:, 0])
ax_patches = fig.add_subplot(grid[0, 1])
ax_flood = fig.add_subplot(grid[1, 1])

ax_map.imshow(np.where(valid, bands['post_vv'], np.nan), cmap='gray', vmin=-25, vmax=0, extent=extent_km)
block_km = dataset_info['block_size'] * transform.a / 1000
for split, colour in SPLIT_COLOURS.items():
    for by, bx in dataset_info['blocks'][split]:
        x_km = (transform.c + bx * transform.a) / 1000
        y_km = (transform.f + (by + dataset_info['block_size']) * transform.e) / 1000
        ax_map.add_patch(Rectangle((x_km, y_km), block_km, block_km, facecolor=colour, alpha=0.35,
                                   edgecolor=colour, linewidth=1.2))
ax_map.set_title('Each 5 km block belongs to one split')
ax_map.set_xlabel('Easting (km, UTM zone 60S)')
ax_map.set_ylabel('Northing (km)')
ax_map.legend(handles=[Patch(facecolor=c, alpha=0.6, edgecolor=c, label=SPLIT_NAMES[s]) for s, c in SPLIT_COLOURS.items()],
              loc='lower left', frameon=True, facecolor=SURFACE, edgecolor=AXIS)

labels = [SPLIT_NAMES[s] for s in SPLIT_COLOURS]
colours = list(SPLIT_COLOURS.values())
for ax, column, fmt, title in ((ax_patches, 'Patches', '{:.0f}', 'Patches per split'),
                               (ax_flood, 'Flooded pixels (%)', '{:.2f}%', 'Share of flooded pixels')):
    values = split_table[column].to_numpy(dtype=float)
    bars = ax.bar(labels, values, color=colours, width=0.6)
    ax.bar_label(bars, labels=[fmt.format(v) for v in values], padding=4, color=INK)
    ax.set_title(title)
    ax.set_ylim(0, values.max() * 1.18)
    ax.grid(axis='y')
    ax.set_axisbelow(True)
    ax.tick_params(axis='x', colors=INK, length=0)
save(fig, '04_spatial_split.png')

## 5. What the Model Sees

Each training example is a 256 × 256 pixel patch (2.56 km across) with four input channels and a label. The inputs are normalised, so 0 is the average value for that channel and ±1 is one standard deviation.

In [ ]:
# -------------------------------------------------------------------------
# 9. Figure: Example Training Patches (model inputs + label)
# -------------------------------------------------------------------------
N_EXAMPLES = 3
input_bands = dataset_info['input_bands']
train_images = sorted(glob.glob(os.path.join(DATA_DIR, 'train', 'images', '*.npy')))
train_masks = sorted(glob.glob(os.path.join(DATA_DIR, 'train', 'masks', '*.npy')))

# The most flooded training patches show the label best
flood_share = np.array([np.load(p).mean() for p in train_masks])
examples = np.argsort(flood_share)[::-1][:N_EXAMPLES]

fig, axes = plt.subplots(N_EXAMPLES, len(input_bands) + 1, figsize=(3.2 * (len(input_bands) + 1), 3.3 * N_EXAMPLES),
                         squeeze=False, constrained_layout=True)
for row, idx in enumerate(examples):
    image, mask = np.load(train_images[idx]), np.load(train_masks[idx])
    for col, name in enumerate(input_bands):
        z = axes[row, col].imshow(image[col], cmap='gray', vmin=-3, vmax=3)
        image_axes(axes[row, col], name if row == 0 else '')
    axes[row, -1].imshow(np.ones_like(mask), cmap=ListedColormap([AXIS]), vmin=0, vmax=1)
    axes[row, -1].imshow(np.where(mask > 0, 1.0, np.nan), cmap=FLOOD_CMAP, interpolation='nearest')
    image_axes(axes[row, -1], 'Label' if row == 0 else '')
    axes[row, -1].text(0.03, 0.04, f'{100 * mask.mean():.1f}% flooded', transform=axes[row, -1].transAxes,
                       color=INK, fontsize=10, bbox=dict(facecolor=SURFACE, edgecolor='none', pad=2))
fig.colorbar(z, ax=axes[:, :len(input_bands)], location='bottom', shrink=0.5, aspect=40,
             label='Normalised value (z-score)')
save(fig, '05_model_inputs.png')

## 6. Normalisation

The four channels start on different scales: VH is about 6 dB darker than VV. Each is shifted and scaled with statistics from the training pixels only, so all four arrive at the model centred on 0 with a similar spread.

In [ ]:
# -------------------------------------------------------------------------
# 10. Figure: Channel Distributions Before & After Normalisation
# -------------------------------------------------------------------------
mean = np.array(dataset_info['normalisation']['mean'])
std = np.array(dataset_info['normalisation']['std'])
sample = np.stack([np.load(p) for p in train_images[::max(1, len(train_images) // 100)]])  # ~100 patches

fig, (ax_raw, ax_norm) = plt.subplots(1, 2, figsize=(14, 4.8), constrained_layout=True)
for i, name in enumerate(input_bands):
    ax_raw.hist(bands[name][valid], bins=np.linspace(-35, 5, 161), histtype='step', color=SERIES[i], label=name)
    ax_norm.hist(sample[:, i].ravel(), bins=np.linspace(-4, 4, 161), histtype='step', color=SERIES[i], label=name)
ax_raw.set_title('Before: backscatter in dB')
ax_raw.set_xlabel('dB')
ax_norm.set_title('After: normalised with training statistics')
ax_norm.set_xlabel('z-score')
for ax in (ax_raw, ax_norm):
    ax.set_ylabel('Pixels')
    ax.grid(axis='y')
    ax.legend(loc='upper left')
save(fig, '06_normalisation.png')

pd.DataFrame({'Training mean (dB)': mean.round(2), 'Training std (dB)': std.round(2)}, index=input_bands)

## 7. Augmentation

During training, each patch is randomly flipped and rotated by a multiple of 90°, with the label transformed the same way. That gives eight versions of every patch, so 623 training patches act like many more. Validation and test patches are never augmented.

In [ ]:
# -------------------------------------------------------------------------
# 11. Figure: The Eight Flip / Rotation Variants of One Patch
# -------------------------------------------------------------------------
post_idx = input_bands.index('post_vv')
image, mask = np.load(train_images[examples[0]])[post_idx], np.load(train_masks[examples[0]])

fig, axes = plt.subplots(2, 4, figsize=(13, 6.8), constrained_layout=True)
for col, k in enumerate(range(4)):
    for row, flip in enumerate((False, True)):
        img, msk = np.rot90(image, k), np.rot90(mask, k)
        if flip:
            img, msk = img[:, ::-1], msk[:, ::-1]
        ax = axes[row, col]
        ax.imshow(img, cmap='gray', vmin=-3, vmax=3)
        ax.contour(msk, levels=[0.5], colors=FLOOD, linewidths=1.5)
        image_axes(ax, f"Rotate {90 * k}°" + (' + flip' if flip else ''))
fig.suptitle('post_vv with the flood label outlined in blue', x=0.01, ha='left', color=INK_2, fontsize=11)
save(fig, '07_augmentation.png')